In [1]:
# Preprocessing Data
import os
import numpy as np
import cv2
from tqdm import tqdm
from scipy import special
import shutil

def k_gamma_function(z, k=2):
    """
    Implements the k-Gamma function: Γ_k(z) = k^(z/k - 1) * Γ(z/k)
    
    Parameters:
    -----------
    z : float
        Input value
    k : float
        The k parameter
        
    Returns:
    --------
    float : k-Gamma function value
    """
    return k**(z/k - 1) * special.gamma(z/k)

def k_mittag_leffler(r, rho, k=2, max_terms=50):
    """
    Computes the k-Mittag-Leffler function as shown in equation (9)
    E^(1,1-ρ/k)_k(r)
    
    Parameters:
    -----------
    r : float
        Pixel probability value (0-1)
    rho : float
        Fractional order parameter
    k : float
        The k parameter
    max_terms : int
        Maximum number of terms in the summation
        
    Returns:
    --------
    float : k-Mittag-Leffler function value
    """
    # STEP 4: Using Eq. (9) to determine the proposed K-CFDO
    # Initialize the summation
    result = 0
    
    # Compute the series as per equation (9)
    for n in range(max_terms):
        # r^n / (n! * Γ_k(1+n-ρ/k))
        numerator = r**n
        denominator = np.math.factorial(n) * k_gamma_function(1 + n - rho/k, k)
        term = numerator / denominator
        result += term
        
        # Stop if terms become very small
        if abs(term) < 1e-10:
            break
    
    # Multiply by r^(1-ρ/k) as in equation (9)
    result *= r**(1 - rho/k)
    
    return result

def calculate_pixel_probability(image):
    """
    Calculate the probability value (r) for each pixel
    For simplicity, normalized pixel intensity is used as probability
    
    Parameters:
    -----------
    image : ndarray
        Input grayscale image
        
    Returns:
    --------
    ndarray : Probability map with same shape as input
    """
    # STEP 3: Determine the pixel's probability value (r)
    # Normalize to [0,1] range
    if image.max() > 1.0:
        normalized = image.astype(np.float64) / 255.0
    else:
        normalized = image.astype(np.float64)
    
    return normalized

def k_cfdo_enhancement_full(image, rho=0.5, k=2, window_size=3):
    """
    Complete implementation of K-CFDO enhancement following all algorithm steps
    
    Parameters:
    -----------
    image : ndarray
        Input grayscale image
    rho : float
        Fractional order parameter
    k : float
        The k parameter
    window_size : int
        Size of the window for neighborhood processing
        
    Returns:
    --------
    ndarray : Enhanced image
    """
    # STEP 1: Consider the source image (already done - input parameter)
    img = image.astype(np.float64)
    
    # STEP 2: Set the parameters of ρ and k (done via function parameters)
    
    # STEP 3: Determine the pixel's probability value (r)
    r_map = calculate_pixel_probability(img)
    
    # Create output image
    n, m = img.shape
    enhanced_img = np.zeros_like(img)
    
    # Pad the image for window processing
    pad_width = window_size // 2
    padded_img = np.pad(img / 255.0, pad_width, mode='reflect')
    padded_r = np.pad(r_map, pad_width, mode='reflect')
    
    # STEP 5: Calculate the enhanced image using Eq. (10)
    # Loop through each pixel in the original image
    for i in range(n):
        for j in range(m):
            # Initialize enhancement value
            enhancement = 0
            
            # Apply the double summation as shown in Eq. (10)
            for di in range(window_size):
                for dj in range(window_size):
                    # Get pixel position in padded image
                    pi = i + di
                    pj = j + dj
                    
                    # Get pixel value and probability
                    pixel_value = padded_img[pi, pj]
                    r_value = padded_r[pi, pj]
                    
                    # Calculate terms from Eq. (10)
                    r_term = r_value**(1 - rho/k)
                    gamma_term = k_gamma_function(2 - rho/k, k)
                    
                    # Add to the enhancement value
                    enhancement += pixel_value * r_term / gamma_term
            
            # Store the enhanced pixel value
            enhanced_img[i, j] = enhancement
    
    # Normalize the enhanced image
    if enhanced_img.min() != enhanced_img.max():
        enhanced_img = (enhanced_img - enhanced_img.min()) / (enhanced_img.max() - enhanced_img.min())
    
    return (enhanced_img * 255).astype(np.uint8)

def process_images(input_dir, output_dir, rho=0.7, k=1.0):
    """
    Process all images in the input directory and save to output directory
    
    Parameters:
    -----------
    input_dir : str
        Path to directory containing input images
    output_dir : str
        Path to directory where enhanced images will be saved
    rho : float
        Fractional order parameter
    k : float
        The k parameter
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get list of image files
    image_files = [f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, f)) and 
                  f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
    
    if not image_files:
        print(f"No images found in {input_dir}")
        return
    
    print(f"Processing {len(image_files)} images with rho={rho}, k={k}...")
    
    # Process each image
    for img_file in tqdm(image_files):
        # Read image
        img_path = os.path.join(input_dir, img_file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            print(f"Could not read image: {img_path}")
            continue
            
        # Enhance image
        enhanced_img = k_cfdo_enhancement_full(img, rho=rho, k=k)
        
        # Save enhanced image
        output_path = os.path.join(output_dir, img_file)
        cv2.imwrite(output_path, enhanced_img)

def copy_labels(input_label_dir, output_label_dir):
    """
    Copy all label files from input directory to output directory
    
    Parameters:
    -----------
    input_label_dir : str
        Path to directory containing input labels
    output_label_dir : str
        Path to directory where labels will be copied
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_label_dir, exist_ok=True)
    
    # If input directory doesn't exist or is not a directory
    if not os.path.exists(input_label_dir) or not os.path.isdir(input_label_dir):
        print(f"Input label directory does not exist: {input_label_dir}")
        return
    
    # Copy all files from input to output
    label_files = [f for f in os.listdir(input_label_dir) if os.path.isfile(os.path.join(input_label_dir, f))]
    
    if not label_files:
        print(f"No label files found in {input_label_dir}")
        return
    
    print(f"Copying {len(label_files)} label files...")
    
    for label_file in tqdm(label_files):
        src_path = os.path.join(input_label_dir, label_file)
        dst_path = os.path.join(output_label_dir, label_file)
        shutil.copy2(src_path, dst_path)

# Main execution
if __name__ == "__main__":
    # Directories
    input_image_dir = "../data/images"
    input_label_dir = "../data/labels"
    output_dir = "../enhanced_data"
    output_image_dir = os.path.join(output_dir, "images")
    output_label_dir = os.path.join(output_dir, "labels")
    
    # Fixed parameters
    rho = 0.7
    k = 1.0
    
    # Process images
    process_images(input_image_dir, output_image_dir, rho=rho, k=k)
    
    # Copy labels
    copy_labels(input_label_dir, output_label_dir)
    
    print("Processing complete!")
    print(f"Enhanced images saved to: {output_image_dir}")
    print(f"Labels copied to: {output_label_dir}")

Processing 200 images with rho=0.7, k=1.0...


100%|██████████████████████████████████████████████████████████████████████████████| 200/200 [4:51:48<00:00, 87.54s/it]


Copying 200 label files...


100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [00:04<00:00, 44.67it/s]

Processing complete!
Enhanced images saved to: ../enhanced_data\images
Labels copied to: ../enhanced_data\labels


In [19]:
# Splitting Data
import os
import random
import shutil
from tqdm import tqdm
import numpy as np
import yaml

def split_dataset(images_dir, labels_dir, output_base_dir, split_ratios=(0.7, 0.2, 0.1), seed=42, class_names=["tooth"]):
    """
    Split the dataset images and corresponding labels into train, validation, and test sets.
    Organizes output inside 'enhanced_data/' with train/valid/test folders and generates data.yaml.
    """

    random.seed(seed)
    np.random.seed(seed)

    if not np.isclose(sum(split_ratios), 1.0):
        raise ValueError("Split ratios must sum to 1.0")

    # Set up output directories
    train_img_dir = os.path.join(output_base_dir, "train", "images")
    valid_img_dir = os.path.join(output_base_dir, "valid", "images")
    test_img_dir = os.path.join(output_base_dir, "test", "images")

    train_label_dir = os.path.join(output_base_dir, "train", "labels")
    valid_label_dir = os.path.join(output_base_dir, "valid", "labels")
    test_label_dir = os.path.join(output_base_dir, "test", "labels")

    for directory in [train_img_dir, valid_img_dir, test_img_dir,
                      train_label_dir, valid_label_dir, test_label_dir]:
        os.makedirs(directory, exist_ok=True)

    image_files = [f for f in os.listdir(images_dir)
                   if os.path.isfile(os.path.join(images_dir, f)) and
                   f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]

    if not image_files:
        print(f"No images found in {images_dir}")
        return

    random.shuffle(image_files)

    num_files = len(image_files)
    train_end = int(num_files * split_ratios[0])
    val_end = train_end + int(num_files * split_ratios[1])

    train_files = image_files[:train_end]
    val_files = image_files[train_end:val_end]
    test_files = image_files[val_end:]

    print(f"Total images: {num_files}")
    print(f"Train: {len(train_files)}")
    print(f"Validation: {len(val_files)}")
    print(f"Test: {len(test_files)}")

    def copy_files(file_list, dst_img_dir, dst_label_dir):
        for filename in tqdm(file_list, desc=f"Copying to {os.path.dirname(dst_img_dir)}"):
            src_img = os.path.join(images_dir, filename)
            dst_img = os.path.join(dst_img_dir, filename)

            shutil.copy2(src_img, dst_img)

            label_name = os.path.splitext(filename)[0] + ".txt"
            src_label = os.path.join(labels_dir, label_name)
            dst_label = os.path.join(dst_label_dir, label_name)

            if os.path.exists(src_label):
                shutil.copy2(src_label, dst_label)

    # Copy files
    copy_files(train_files, train_img_dir, train_label_dir)
    copy_files(val_files, valid_img_dir, valid_label_dir)
    copy_files(test_files, test_img_dir, test_label_dir)

    # Create YOLOv8 data.yaml
    yaml_dict = {
        'path': output_base_dir,
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'names': class_names
    }

    yaml_path = os.path.join(output_base_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_dict, f, default_flow_style=False)

    print(f"\n✅ Dataset split complete and saved in: {output_base_dir}")
    print(f"✅ YAML file created at: {yaml_path}")

if __name__ == "__main__":
    base_dir = "../enhanced_data"
    images_dir = os.path.join(base_dir, "images")
    labels_dir = os.path.join(base_dir, "labels")

    split_dataset(images_dir, labels_dir, base_dir, split_ratios=(0.7, 0.2, 0.1))

Total images: 200
Train: 140
Validation: 40
Test: 20


Copying to ../enhanced_data\test: 100%|████████████████████████████████████████████████| 20/20 [00:00<00:00, 67.35it/s]


✅ Dataset split complete and saved in: ../enhanced_data
✅ YAML file created at: ../enhanced_data\data.yaml


In [23]:
# Tooth Segmentation – Training a YOLOv8 segmentation model.
'''
    This script trains a YOLOv8 segmentation model to automatically segment individual teeth from dental X-ray images. 
    The model is designed for instance segmentation, which detects objects (teeth) and predicts their pixel-wise masks. 
    Using a dataset of labeled X-rays, the model learns to identify and separate each tooth.
    Key training parameters like optimizer, learning rate, and momentum are adjusted for optimal performance.
    
    The script uses the nano (n) version of YOLOv8, which is the smallest and fastest model variant,
    making it suitable for mobile deployment while still maintaining acceptable segmentation quality.
    
    After training, the script includes an export step to convert the model to ONNX format,
    which can be used for efficient inference on mobile devices.
'''
from ultralytics import YOLO
from tensorflow.keras.callbacks import TensorBoard

# Load the YOLOv8 instance segmentation model (the segmentation model)
# Using YOLOv8n-seg which is the smallest variant, ideal for mobile applications
model = YOLO("yolov8n-seg.pt")  # Make sure you use the segmentation variant

# Train the model on the instance segmentation dataset
model.train(
    data="C:/Users/pauli/DeepDent/Periodont/enhanced_data/data.yaml",  # Path to the data.yaml file
    epochs=100,                        # Number of epochs
    batch=8,                          # Batch size
    device="cpu",                     # Use CPU for training (or "cuda" if you have a GPU)
    
    # Optimization parameters
    optimizer="AdamW",                # Use AdamW optimizer (default is Adam)
    lr0=0.001,                        # Initial learning rate
    momentum=0.937,                   # Momentum for the optimizer
    weight_decay=0.0005,              # Regularization term
    warmup_epochs=3,                  # Number of epochs for warmup
    warmup_momentum=0.8,              # Momentum during warmup
    warmup_bias_lr=0.1,               # Bias learning rate during warmup
    
    # CPU-specific optimizations
    workers=4,                        # Number of worker threads for data loading
    cache=False,                      # Don't cache images in RAM (helps with CPU training)
    plots=True,                       # Generate performance plots to track progress
    
    # Segmentation improvements
    overlap_mask=True,                # Improves mask quality for overlapping teeth
    mask_ratio=4,                     # Controls mask resolution (higher = more detailed masks)
    iou=0.7,                          # IoU threshold for NMS (higher = stricter box matching)
    
    # Loss function adjustments for better dental segmentation
    box=0.5,                          # Box loss weight
    cls=0.5,                          # Classification loss weight 
    dfl=1.0,                          # Distribution focal loss weight
    
    # Validation and saving parameters
    val=True,                         # Run validation during training
    save=True,                        # Save model checkpoints
    save_period=5                     # Save model every 5 epochs
)

New https://pypi.org/project/ultralytics/8.3.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.99  Python-3.9.13 torch-2.6.0+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
engine\trainer: task=segment, mode=train, model=yolov8n-seg.pt, data=C:/Users/pauli/DeepDent/Periodont/enhanced_data/data.yaml, epochs=100, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=5, cache=False, device=cpu, workers=4, project=None, name=train13, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina

train: Scanning C:\Users\pauli\DeepDent\Periodont\enhanced_data\train\labels.cache... 160 images, 1 backgrounds, 0 corr

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning C:\Users\pauli\DeepDent\Periodont\enhanced_data\valid\labels.cache... 40 images, 0 backgrounds, 0 corrupt

module 'matplotlib.cm' has no attribute 'register_cmap'
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         40       1057      0.987       0.98      0.993      0.749      0.983      0.976      0.992       0.63
WARNING  ConfusionMatrix plot failure: module 'matplotlib.cm' has no attribute 'register_cmap'
WARNING  ConfusionMatrix plot failure: module 'matplotlib.cm' has no attribute 'register_cmap'
Speed: 2.7ms preprocess, 97.6ms inference, 0.0ms loss, 62.6ms postprocess per image
Results saved to runs\segment\train13


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001B1A3961520>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0410

In [37]:
from ultralytics import YOLO

# Load trained segmentation model
model = YOLO('runs/segment/train13/weights/best.pt')

# Validate the model
results = model.val(
    data="C:/Users/pauli/DeepDent/Periodont/enhanced_data/data.yaml",
    imgsz=640,
    batch=8,
    device="cpu"
)

# Extract metrics
map50 = results.seg.map50
map5095 = results.seg.map
mp = results.seg.mp
mr = results.seg.mr

# Calculate F1 score
f1 = 2 * (mp * mr) / (mp + mr + 1e-6)

# Calculate IoU approximation
iou = (mp * mr) / (mp + mr - mp * mr + 1e-6)

# Print the evaluation
print("\n--- Segmentation Model Evaluation ---")
print(f"Segmentation mAP50       : {map50:.4f}")
print(f"Segmentation mAP50-95     : {map5095:.4f}")
print(f"Segmentation Mean Precision : {mp:.4f}")
print(f"Segmentation Mean Recall    : {mr:.4f}")
print(f"Segmentation F1 Score       : {f1:.4f}")
print(f"Segmentation IoU            : {iou:.4f}")


Ultralytics 8.3.99  Python-3.9.13 torch-2.6.0+cpu CPU (AMD Ryzen 5 5500U with Radeon Graphics)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs


val: Scanning C:\Users\pauli\DeepDent\Periodont\enhanced_data\valid\labels.cache... 40 images, 0 backgrounds, 0 corrupt
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...
WARNING  Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         40       1057      0.987       0.98      0.993      0.749      0.983      0.976      0.992       0.63
WARNING  ConfusionMatrix plot failure: module 'matplotlib.cm' has no attribute 'register_cmap'
WARNING  ConfusionMatrix plot failure: module 'matplotlib.cm' has no attribute 'register_cmap'
Speed: 1.2ms preprocess, 78.1ms inference, 0.0ms loss, 35.2ms postprocess per image
Results saved to runs\segment\val10

--- Segmentation Model Evaluation ---
Segmentation mAP50       : 0.9920
Segmentation mAP50-95     : 0.6299
Segmentation Mean Precision : 0.9833
Segmentation Mean Recall    : 0.9763
Segmentation F1 Score       : 0.9798
Segmentation IoU            : 0.9604


In [39]:
pip install torch tensorflow tf2onnx onnx

  Using cached tf2onnx-1.16.1-py3-none-any.whl (455 kB)
  Using cached protobuf-5.29.4-cp39-cp39-win_amd64.whl (434 kB)
  Using cached protobuf-3.20.3-cp39-cp39-win_amd64.whl (904 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.30.2
    Uninstalling protobuf-6.30.2:
      Successfully uninstalled protobuf-6.30.2
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\pauli\\anaconda3\\Lib\\site-packages\\google\\~=pb\\_message.cp39-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [44]:
import torch
from ultralytics import YOLO  # Ensure you have ultralytics installed

# Load the YOLOv8 model correctly
model = YOLO('runs/segment/train13/weights/best.pt')  # Load the model using YOLO class from ultralytics

# Ensure the model is in evaluation mode
model.eval()

# Create a dummy input for the model (adjust size if necessary)
dummy_input = torch.randn(1, 3, 640, 640)

# Export the model to ONNX format
onnx_model_path = '../model/yolov8_model.onnx'
torch.onnx.export(model.model, dummy_input, onnx_model_path, opset_version=12)


In [46]:
pip install onnx-tf

Note: you may need to restart the kernel to use updated packages.


In [56]:
pip install tensorflow==2.8.0

     -------------------------------------- 438.0/438.0 MB 2.4 MB/s eta 0:00:00
     -------------------------------------- 462.5/462.5 kB 4.1 MB/s eta 0:00:00
     ---------------------------------------- 5.8/5.8 MB 3.9 MB/s eta 0:00:00
     ---------------------------------------- 1.4/1.4 MB 3.2 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.9.0
    Uninstalling keras-2.9.0:
      Successfully uninstalled keras-2.9.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.9.1
    Uninstalling tensorboard-2.9.1:
      Successfully uninstalled tensorboard-2.9.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.9.0
    Uninstalling tensorflow-2.9.0:
      Successfully uninstalled tensorflow-2.9.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.8.0 which is incompatible.
tensorflow-intel 2.17.0 requires flatbuffers>=24.3.25, but you have flatbuffers 1.12 which is incompatible.
tensorflow-intel 2.17.0 requires keras>=3.2.0, but you have keras 2.8.0 which is incompatible.
tensorflow-intel 2.17.0 requires ml-dtypes<0.5.0,>=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 3.19.6 which is incompatible.
tensorflow-intel 2.17.0 requires tensorboard<2.18,>=2.17, but you have tensorboard 2.8.0 which is incompatible.


In [57]:
pip install onnx-tensorflow

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement onnx-tensorflow (from versions: none)
ERROR: No matching distribution found for onnx-tensorflow


In [60]:
pip install --upgrade tensorflow-addons

  Using cached tensorflow_addons-0.22.0-cp39-cp39-win_amd64.whl (729 kB)
  Attempting uninstall: tensorflow-addons
    Found existing installation: tensorflow-addons 0.18.0
    Uninstalling tensorflow-addons-0.18.0:
      Successfully uninstalled tensorflow-addons-0.18.0
Note: you may need to restart the kernel to use updated packages.


In [61]:
import onnx
from onnx_tf.backend import prepare

# Load the ONNX model
onnx_model_path = '../model/yolov8_model.onnx'
onnx_model = onnx.load(onnx_model_path)

# Convert the ONNX model to TensorFlow
tf_rep = prepare(onnx_model)

# Export the TensorFlow model as a SavedModel
saved_model_path = 'yolov8_indiv_teeth_seg_model'
tf_rep.export_graph(saved_model_path)

C:\Users\pauli\anaconda3\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


AttributeError: module 'keras._tf_keras.keras.layers' has no attribute 'AbstractRNNCell'